# DeepEval — Offline & Online evals (Azure OpenAI judge)

This notebook is split into **two parts that share the same setup and the same metrics**:

- **Part 1 — Offline evals:** the eval data lives in **files generated inside this Colab** (a **PDF** policy document and an **Excel** dataset). We load those files and score them.
- **Part 2 — Online evals:** the eval data comes from a **JSON** file (e.g. live / production logs). The plumbing is in place but the JSON is intentionally **left empty for now**.

Both parts reuse one shared judge model and one shared metric set, so the only thing that changes between offline and online is *where the data comes from*.

**Metrics used (6, spanning DeepEval's families):** `ContextualRelevancyMetric` (RAG retriever), `AnswerRelevancyMetric` + `FaithfulnessMetric` (RAG generator), `GEval` (custom), `ToxicityMetric` (safety), `TurnRelevancyMetric` (multi-turn).

API usage follows the current docs at https://deepeval.com/docs/introduction.

# Shared setup

## Install dependencies

`deepeval` (core) + `langchain-openai` (the Azure chatbot under test) + `pandas`/`openpyxl` (Excel I/O) + `fpdf2`/`pypdf` (make & read the PDF) + `tabulate`.

In [ ]:
%pip install -q deepeval langchain-openai pandas openpyxl fpdf2 pypdf tabulate
# On a version conflict, try:  %pip install -U deepeval

In [ ]:
!deepeval --version

## Configure the Azure OpenAI judge

> **Security note:** keys are inlined so the notebook runs as-is. Hard-coded secrets are a leak risk — **rotate these** and load them from Colab secrets / `getpass` for anything beyond a local demo.

In [ ]:
import os

os.environ["AZURE_API_KEY"] = "5xD6IeVNWL7eYqgbuxRKVATJK38hdacwDJnVT8rbKH9W1Kx6NhZZJQQJ99CDACYeBjFXJ3w3AAABACOGD59K"
# base_url = RESOURCE root (NOT the full /chat/completions path); the SDK appends the route.
os.environ["AZURE_API_BASE"] = "https://tcoeaiteamgpt41nano.openai.azure.com/"
os.environ["AZURE_API_VERSION"] = "2024-12-01-preview"
os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"] = "gpt-4.1-nano"
os.environ["AZURE_MODEL_NAME"] = "gpt-4.1-nano"

# Optional — only needed for the Confident AI UI (Tracing section).
os.environ["CONFIDENT_API_KEY"] = "confident_us_org_uGYD76TLx8bayz80vrO5Y2xrSokvU+wwAcxBBaw4vVM="

## Build the judge model and the app-under-test

`azure_model` is the **judge** (passed to every metric / the synthesizer). `chatbot` is the **app under test** — it produces the `actual_output` we grade. (Ref: https://deepeval.com/integrations/models/azure-openai)

In [ ]:
from deepeval.models import AzureOpenAIModel
from langchain_openai import AzureChatOpenAI

# Judge: stable, deterministic scoring.
azure_model = AzureOpenAIModel(
    model=os.environ["AZURE_MODEL_NAME"],
    deployment_name=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_key=os.environ["AZURE_API_KEY"],
    api_version=os.environ["AZURE_API_VERSION"],
    base_url=os.environ["AZURE_API_BASE"],
    temperature=0,
)

# App under test.
chatbot = AzureChatOpenAI(
    azure_deployment=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_key=os.environ["AZURE_API_KEY"],
    openai_api_version=os.environ["AZURE_API_VERSION"],
    azure_endpoint=os.environ["AZURE_API_BASE"],
    temperature=0.3,
)

print("Judge:", azure_model.get_model_name())
print("Sanity check ->", chatbot.invoke("Reply with the single word: ok").content)

## Define the shared metrics + a reusable runner

Defined **once** here and reused by both the offline and online parts. Each metric returns a score in `[0, 1]` plus a reason and passes when `score >= threshold` (default `0.5`). (Refs: https://deepeval.com/docs/metrics-introduction)

| Metric | Family | Measures |
|---|---|---|
| `ContextualRelevancyMetric` | RAG · retriever | is the retrieved context relevant to the question? |
| `AnswerRelevancyMetric` | RAG · generator | does the answer address the question? |
| `FaithfulnessMetric` | RAG · generator | is the answer grounded in the context (no hallucination)? |
| `GEval` | custom | a plain-English criterion (here: correctness vs. expected) |
| `ToxicityMetric` | safety | is the output toxic/harmful? |

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    ContextualRelevancyMetric,
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ToxicityMetric,
    GEval,
)
from deepeval.test_case import LLMTestCaseParams

def build_metrics():
    """Fresh metric instances (metrics hold state after measure())."""
    correctness = GEval(
        name="Correctness",
        model=azure_model,
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        evaluation_steps=[
            "Check whether facts in 'actual output' contradict any facts in 'expected output'.",
            "Lightly penalise missing important detail; focus on the main idea.",
            "Vague wording or differing opinions are acceptable.",
        ],
    )
    return [
        ContextualRelevancyMetric(model=azure_model),
        AnswerRelevancyMetric(model=azure_model),
        FaithfulnessMetric(model=azure_model),
        ToxicityMetric(model=azure_model),
        correctness,
    ]

def run_eval(test_cases, label=""):
    if not test_cases:
        print(f"[{label}] no test cases - skipping.")
        return None
    print(f"[{label}] evaluating {len(test_cases)} test case(s)...")
    return evaluate(test_cases=test_cases, metrics=build_metrics())

# Part 1 — Offline evals

The eval data is materialised as **files inside this Colab**. We create two offline artifacts:
1. a **PDF** policy document (`hr_policy.pdf`), and
2. an **Excel** eval dataset (`offline_hr_dataset.xlsx`) — synthetic goldens generated from the policy.

Then we load the Excel back, generate answers with the chatbot, and score them.

## 1.1 Generate the offline source data in Colab

One source of truth (`POLICY`) is rendered to a **PDF** and also used as `contexts` for the `Synthesizer`. The PDF is a genuine offline document you could swap for any real policy file.

In [ ]:
POLICY = [
    ("Maternity Leave", [
        "Female employees are entitled to 26 weeks of paid maternity leave under the Maternity Benefit Act.",
        "Up to 8 weeks of maternity leave may be taken before the expected date of delivery.",
    ]),
    ("Dress Code", [
        "On casual Fridays employees may wear clean jeans and simple t-shirts.",
        "Clothing with rips, slogans, or political prints is not permitted.",
    ]),
    ("Earned Leave", [
        "Employees accrue 1.5 days of earned leave per month, capped at 18 days a year.",
        "Leave requests must be approved through the HR portal at least 3 days in advance.",
    ]),
]

contexts = [facts for _title, facts in POLICY]   # list[list[str]] for the synthesizer

In [ ]:
# --- Render POLICY to a PDF (offline document artifact) ---
from fpdf import FPDF

PDF_PATH = "hr_policy.pdf"
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 12, "Company HR Policy", ln=True)
for title, facts in POLICY:
    pdf.ln(4)
    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 9, title, ln=True)
    pdf.set_font("Helvetica", "", 11)
    for fact in facts:
        pdf.multi_cell(0, 7, f"- {fact}")
pdf.output(PDF_PATH)
print("Wrote", PDF_PATH)

# Quick read-back to confirm the PDF has an extractable text layer.
from pypdf import PdfReader
print(PdfReader(PDF_PATH).pages[0].extract_text()[:200], "...")

### Build the Excel eval dataset from the policy

`generate_goldens_from_contexts` synthesizes `(input, expected_output, context)` goldens **without needing an embedding model**. We wrap them in an `EvaluationDataset`, save natively to CSV via `save_as`, and export the **Excel** file we reload below. (Refs: https://deepeval.com/docs/synthesizer-generate-from-contexts, https://deepeval.com/docs/evaluation-datasets)

In [ ]:
import json
import pandas as pd
from deepeval.synthesizer import Synthesizer
from deepeval.dataset import EvaluationDataset

synthesizer = Synthesizer(model=azure_model)
goldens = synthesizer.generate_goldens_from_contexts(
    contexts=contexts,
    include_expected_output=True,
    max_goldens_per_context=2,   # ~6 goldens total
)

dataset = EvaluationDataset(goldens=goldens)
os.makedirs("deepeval_dataset", exist_ok=True)
dataset.save_as(file_type="csv", directory="deepeval_dataset", file_name="hr_goldens")  # native CSV

EXCEL_PATH = "offline_hr_dataset.xlsx"
pd.DataFrame([{
    "input": g.input,
    "expected_output": g.expected_output,
    "context": json.dumps(g.context),   # list -> JSON string for the cell
} for g in goldens]).to_excel(EXCEL_PATH, index=False)
print(f"Wrote {EXCEL_PATH} ({len(goldens)} rows)")
pd.read_excel(EXCEL_PATH).head()

> **Optional — synthesize straight from the PDF.** The canonical "from a document" path is `generate_goldens_from_docs`. It chunks + embeds the file, so it needs an **embedding model** (default OpenAI; configure an Azure embedding deployment to keep it all on Azure). Guarded so it won't break the run:

In [ ]:
try:
    doc_goldens = Synthesizer(model=azure_model).generate_goldens_from_docs(
        document_paths=[PDF_PATH],
        include_expected_output=True,
    )
    print(f"Generated {len(doc_goldens)} goldens from the PDF.")
except Exception as e:
    print("Skipped from-docs synthesis (needs an embedding model configured):")
    print(" ", type(e).__name__, "-", e)

## 1.2 Load the offline Excel → build test cases

Read the **same** Excel back, run each question through the chatbot for a real `actual_output`, and assemble `LLMTestCase`s. The golden's `context` doubles as the simulated `retrieval_context`.

In [ ]:
import json
import pandas as pd
from deepeval.test_case import LLMTestCase

SYSTEM = ("You are an HR assistant. Answer the employee question using ONLY the policy facts provided. "
          "Be concise.")

def build_offline_test_cases(excel_path):
    rows = pd.read_excel(excel_path)
    cases = []
    for _, row in rows.iterrows():
        context = json.loads(row["context"])
        prompt = f"{SYSTEM}\n\nPolicy facts:\n- " + "\n- ".join(context) + f"\n\nQuestion: {row['input']}"
        cases.append(LLMTestCase(
            input=row["input"],
            actual_output=chatbot.invoke(prompt).content,
            expected_output=row["expected_output"],
            retrieval_context=context,
            context=context,
        ))
    return cases

offline_cases = build_offline_test_cases(EXCEL_PATH)
print(f"Built {len(offline_cases)} offline test cases.")
print("Q:", offline_cases[0].input)
print("A:", offline_cases[0].actual_output)

## 1.3 Run the offline single-turn evaluation

Score the offline test cases with the shared metric set.

In [ ]:
offline_results = run_eval(offline_cases, label="offline")

### Debug a single metric (`measure()` + `verbose_mode`)

Call one metric directly to see the judge's intermediate reasoning.

In [ ]:
m = AnswerRelevancyMetric(model=azure_model, verbose_mode=True)
m.measure(offline_cases[0])
print("\nscore :", m.score)
print("reason:", m.reason)

## 1.4 Offline multi-turn evaluation

Conversations are scored with a `ConversationalTestCase` of `Turn`s. Here `TurnRelevancyMetric` on a good vs. a bad dress-code conversation. (Ref: https://deepeval.com/docs/metrics-introduction)

In [ ]:
from deepeval.test_case import ConversationalTestCase, Turn
from deepeval.metrics import TurnRelevancyMetric

good_convo = ConversationalTestCase(
    chatbot_role="dress code advisor",
    scenario="User asks about dress code policy",
    expected_outcome="Assistant answers only what was asked, nothing extra.",
    turns=[
        Turn(role="user", content="Can I wear hoodies to the company?"),
        Turn(role="assistant", content="Hoodies are generally considered unprofessional, so it is best to avoid them."),
        Turn(role="user", content="Are t-shirts fine?"),
        Turn(role="assistant", content="Yes, as long as they look professional."),
    ],
)
bad_convo = ConversationalTestCase(
    chatbot_role="dress code advisor",
    scenario="User asks about dress code policy",
    expected_outcome="Assistant answers only what was asked, nothing extra.",
    turns=[
        Turn(role="user", content="Can I wear hoodies to the company?"),
        Turn(role="assistant", content="I am not sure whether blazers can be worn."),
        Turn(role="user", content="Are t-shirts fine?"),
        Turn(role="assistant", content="Yes, hoodies are professional."),
    ],
)

evaluate(
    test_cases=[good_convo, bad_convo],
    metrics=[TurnRelevancyMetric(model=azure_model, include_reason=True)],
)

# Part 2 — Online evals (JSON)  ·  *left empty for now*

Same judge, same metrics — only the **data source** differs. Online data arrives as a **JSON** file (e.g. exported production traffic). The loader and the eval call are wired up; the JSON is intentionally empty, so this section runs cleanly and does nothing until you drop records in.

**Expected JSON schema** (a list of records):
```json
[
  {
    "input": "employee question",
    "actual_output": "the answer your live app returned",
    "expected_output": "ground truth (optional)",
    "retrieval_context": ["context chunk 1", "context chunk 2"]
  }
]
```

In [ ]:
import json

ONLINE_JSON_PATH = "online_data.json"

# Create an EMPTY online dataset placeholder (replace [] with real records later).
if not os.path.exists(ONLINE_JSON_PATH):
    with open(ONLINE_JSON_PATH, "w") as f:
        json.dump([], f)
print("Online data file:", ONLINE_JSON_PATH)

In [ ]:
from deepeval.test_case import LLMTestCase

def build_online_test_cases(json_path):
    """Read live/production records from JSON into LLMTestCases."""
    with open(json_path) as f:
        records = json.load(f)
    cases = []
    for r in records:
        cases.append(LLMTestCase(
            input=r["input"],
            actual_output=r["actual_output"],
            expected_output=r.get("expected_output"),
            retrieval_context=r.get("retrieval_context"),
            context=r.get("context"),
        ))
    return cases

online_cases = build_online_test_cases(ONLINE_JSON_PATH)
online_results = run_eval(online_cases, label="online")   # no-op until the JSON has data

# Shared extras

## Tracing & the Confident AI UI

DeepEval can stream traces to the Confident AI dashboard. Configure the environment, then wrap any function with `@observe()`. (Ref: https://deepeval.com/docs/evaluation-llm-tracing)

In [ ]:
from deepeval.tracing import observe, trace_manager

trace_manager.configure(environment="production")  # development | staging | production | testing

@observe()
def graded_answer(test_case):
    m = AnswerRelevancyMetric(model=azure_model)
    m.measure(test_case)
    return {"score": m.score, "reason": m.reason}

print(graded_answer(offline_cases[0]))
# With CONFIDENT_API_KEY set (and `deepeval login`), this trace shows up in the UI.

## The rest of DeepEval — reference map

What we didn't code, with entry points to extend the notebook:

**Synthesizer — other modes:** `generate_goldens_from_scratch(num_goldens=N)` (no source data, drive with `StylingConfig`), `generate_goldens_from_goldens(...)` (augment), plus `EvolutionConfig` / `FiltrationConfig` for complexity & quality. The **Conversation Simulator** synthesizes multi-turn dialogues.

**More metrics:** *Custom* `DAGMetric` (deterministic), `ConversationalGEval`, `ArenaGEval`; *RAG retriever* `ContextualPrecisionMetric`, `ContextualRecallMetric`; *Agentic* `TaskCompletionMetric`, `ToolCorrectnessMetric` (via tracing); *Multi-turn* `KnowledgeRetentionMetric`, `RoleAdherenceMetric`, `ConversationCompletenessMetric`; *Safety* `BiasMetric`, `PIILeakageMetric`, `MisuseMetric`; *Other* `HallucinationMetric`, `SummarizationMetric`, `JsonCorrectnessMetric`, `RagasMetric`; *Multimodal* image metrics via `MLLMTestCase` + `MLLMImage`.

**Other capabilities:** component-level evals (`@observe` + per-span `metrics`), unit testing in CI/CD (`assert_test` + `deepeval test run`), benchmarks (MMLU/HellaSwag/...), prompt optimization, and red teaming via [DeepTeam](https://www.trydeepteam.com/) (`pip install deepteam`).

## Appendix — CLI commands

```bash
# Set Azure OpenAI as the global judge (alternative to AzureOpenAIModel in code)
deepeval set-azure-openai --base-url=<endpoint> --api-key=<key> \
  --deployment-name=<deployment> --model=<model_name> --api-version=2024-12-01-preview
deepeval unset-azure-openai          # revert to OpenAI
deepeval login                       # stream evaluate()/traces to Confident AI
deepeval test run test_llm_app.py    # CI-style unit evals
```